In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import time
import os
from datetime import datetime, timedelta

In [2]:
NIFTY500_TICKERS = {
    # Banking & Finance
    "HDFCBANK.NS":   "Banking",
    "ICICIBANK.NS":  "Banking",
    "KOTAKBANK.NS":  "Banking",
    "AXISBANK.NS":   "Banking",
    "SBIN.NS":       "Banking",
    "INDUSINDBK.NS": "Banking",
    "BANDHANBNK.NS": "Banking",
    "FEDERALBNK.NS": "Banking",
    "BAJFINANCE.NS": "NBFC",
    "BAJAJFINSV.NS": "NBFC",
    "CHOLAFIN.NS":   "NBFC",
    "MUTHOOTFIN.NS": "NBFC",
    "HDFCLIFE.NS":   "Insurance",
    "SBILIFE.NS":    "Insurance",
    "ICICIPRULI.NS": "Insurance"
}

In [ ]:
INDEX_URLS = {
    "Nifty 50":          "ind_nifty50list.csv",
    "Nifty 500":         "ind_nifty500list.csv",
    "Nifty Midcap 150":  "ind_niftymidcap150list.csv",
    "Nifty Smallcap 250":"ind_niftysmallcap250list.csv",
    "Nifty Midcap 50":   "ind_niftymidcap50list.csv",
    "Nifty Next 50":     "ind_niftynext50list.csv",
    "Nifty Bank":        "ind_niftybanklist.csv",
    "Nifty IT":          "ind_niftyittlist.csv",
}

In [3]:
def download_data(tickers, period="14mo"):
    """
    Download OHLCV data for all tickers.
    Returns dict of {ticker: DataFrame}
    """
    print(f"\n{'═'*60}")
    print(f"  Downloading data for {len(tickers)} stocks...")
    print(f"{'═'*60}")

    data = {}
    failed = []

    # Download in batches of 20 to avoid rate limits
    ticker_list = list(tickers.keys())
    batch_size  = 20

    for i in range(0, len(ticker_list), batch_size):
        batch = ticker_list[i:i+batch_size]
        print(f"  Batch {i//batch_size + 1}: {', '.join([t.replace('.NS','') for t in batch])}")

        try:
            raw = yf.download(
                batch,
                period=period,
                auto_adjust=True,
                progress=False,
                threads=True,
            )

            for ticker in batch:
                try:
                    if len(batch) == 1:
                        df = raw.copy()
                    else:
                        df = raw.xs(ticker, axis=1, level=1).copy()

                    df = df.dropna(subset=["Close"])

                    if len(df) < 200:
                        failed.append(ticker)
                        continue

                    data[ticker] = df

                except Exception:
                    failed.append(ticker)

        except Exception as e:
            print(f"  Batch failed: {e}")
            failed.extend(batch)

        time.sleep(0.5)  # be polite to Yahoo Finance

    print(f"\n  Downloaded: {len(data)} stocks  |  Failed: {len(failed)}")
    if failed:
        print(f"  Failed tickers: {', '.join([t.replace('.NS','') for t in failed[:10]])}")

    return data

In [4]:
data = download_data(NIFTY500_TICKERS)


════════════════════════════════════════════════════════════
════════════════════════════════════════════════════════════
  Batch 1: HDFCBANK, ICICIBANK, KOTAKBANK, AXISBANK, SBIN, INDUSINDBK, BANDHANBNK, FEDERALBNK, BAJFINANCE, BAJAJFINSV, CHOLAFIN, MUTHOOTFIN, HDFCLIFE, SBILIFE, ICICIPRULI

  Downloaded: 15 stocks  |  Failed: 0


In [8]:
data['HDFCBANK.NS']

Price,Close,High,Low,Open,Volume
Date,,,,,
2025-02-24,827.366455,830.869098,819.473149,829.339782,20693646
2025-02-25,829.956482,832.743818,824.406501,824.406501,24138708
2025-02-27,839.009155,843.967163,830.055234,830.055234,35786800
2025-02-28,854.647766,857.040413,831.263834,831.263834,54516728
2025-03-03,839.428467,859.901730,835.753110,858.298402,21205364
...,...,...,...,...,...
2026-04-20,795.450012,809.950012,790.000000,797.099976,52148044
2026-04-21,811.750000,814.900024,795.200012,795.200012,49135752
2026-04-22,799.900024,813.049988,798.000000,810.000000,23876412


In [ ]:
INDEX_URLS = {
    "Nifty 50":          "ind_nifty50list.csv",
    "Nifty 500":         "ind_nifty500list.csv",
    "Nifty Midcap 150":  "ind_niftymidcap150list.csv",
    "Nifty Smallcap 250":"ind_niftysmallcap250list.csv",
    "Nifty Midcap 50":   "ind_niftymidcap50list.csv",
    "Nifty Next 50":     "ind_niftynext50list.csv",
    "Nifty Bank":        "ind_niftybanklist.csv",
    "Nifty IT":          "ind_niftyittlist.csv",
}

BASE = "https://archives.nseindia.com/content/indices/"

def get_index_tickers(index_name):
    url = BASE + INDEX_URLS[index_name]
    df  = pd.read_csv(url)
    return [s + ".NS" for s in df['Symbol'].tolist()]

# Example
tickers = get_index_tickers("Nifty Smallcap 250")
print(f"Got {len(tickers)} tickers")

In [17]:
tickers

['ACMESOLAR.NS',
 'AADHARHFC.NS',
 'AARTIIND.NS',
 'AAVAS.NS',
 'ACE.NS',
 'ACUTAAS.NS',
 'ABFRL.NS',
 'ABLBL.NS',
 'ABREL.NS',
 'ABSLAMC.NS',
 'CPPLUS.NS',
 'AEGISLOG.NS',
 'AEGISVOPAK.NS',
 'AFCONS.NS',
 'AFFLE.NS',
 'ABDL.NS',
 'ARE&M.NS',
 'AMBER.NS',
 'ANANDRATHI.NS',
 'ANANTRAJ.NS',
 'ANGELONE.NS',
 'ANURAS.NS',
 'APTUS.NS',
 'ASAHIINDIA.NS',
 'ASTERDM.NS',
 'ATHERENERG.NS',
 'ATUL.NS',
 'BEML.NS',
 'BLS.NS',
 'BALRAMCHIN.NS',
 'BANDHANBNK.NS',
 'BATAINDIA.NS',
 'BAYERCROP.NS',
 'BELRISE.NS',
 'BIKAJI.NS',
 'BSOFT.NS',
 'BLUEDART.NS',
 'BLUEJET.NS',
 'BBTC.NS',
 'FIRSTCRY.NS',
 'BRIGADE.NS',
 'MAPMYINDIA.NS',
 'CCL.NS',
 'CESC.NS',
 'CANFINHOME.NS',
 'CANHLIFE.NS',
 'CAPLIPOINT.NS',
 'CGCL.NS',
 'CARBORUNIV.NS',
 'CARTRADE.NS',
 'CASTROLIND.NS',
 'CEATLTD.NS',
 'CEMPRO.NS',
 'CENTRALBK.NS',
 'CDSL.NS',
 'CHALET.NS',
 'CHAMBLFERT.NS',
 'CHENNPETRO.NS',
 'CHOICEIN.NS',
 'CHOLAHLDNG.NS',
 'CUB.NS',
 'CLEAN.NS',
 'COHANCE.NS',
 'CAMS.NS',
 'CONCORDBIO.NS',
 'CRAFTSMAN.NS',
 'CREDITAC

In [14]:
stock = yf.Ticker("RELIANCE.NS")

info = stock.info 

In [15]:
info

{'address1': 'Maker Chambers IV',
 'address2': '3rd Floor 222 Nariman Point',
 'city': 'Mumbai',
 'zip': '400021',
 'country': 'India',
 'phone': '91 22 3555 5000',
 'fax': '91 22 2204 2268',
 'website': 'https://www.ril.com',
 'industry': 'Oil & Gas Refining & Marketing',
 'industryKey': 'oil-gas-refining-marketing',
 'industryDisp': 'Oil & Gas Refining & Marketing',
 'sector': 'Energy',
 'sectorKey': 'energy',
 'sectorDisp': 'Energy',
 'longBusinessSummary': 'Reliance Industries Limited engages in the hydrocarbon exploration and production, oil and chemicals, retail, and digital service businesses worldwide. It operates through Oil to Chemicals, Oil and Gas, Retail, Digital Services, and Others segments. The company offers refining and marketing products, including liquefied petroleum gas, propylene, naphtha, gasoline, jet/aviation turbine fuel, kerosine oil, diesel, sulphur, and petroleum coke. It also provides polymers, including high-density and low-density polyethylene (PE), line